## Example 1: Basic FastAPI Application Setup

In [ ]:
# Example 1: Basic FastAPI Application Setup
print("FastAPI - Basic Application Setup")

from fastapi import FastAPI

# Create FastAPI application instance
app = FastAPI(
    title="Sample API",
    description="A basic FastAPI example",
    version="1.0.0"
)

# Define basic routes
@app.get("/")
async def read_root():
    return {"message": "Welcome to FastAPI", "version": "1.0.0"}

@app.get("/health")
async def health_check():
    return {"status": "healthy", "service": "api"}

# Show app properties
print(f"✅ FastAPI app created: {app.title} v{app.version}")
print(f"Available routes: {[route.path for route in app.routes if hasattr(route, 'path')]}")
print("Example 1 completed")

## Example 2: Pydantic Models and Request/Response Validation

In [ ]:
# Example 2: Pydantic Models and Request/Response Validation
print("FastAPI - Pydantic Models and Validation")

from fastapi import FastAPI
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional

class User(BaseModel):
    name: str
    age: int
    email: Optional[str] = None
    
    @field_validator('age')
    @classmethod
    def validate_age(cls, v):
        if v < 0 or v > 120:
            raise ValueError('Age must be between 0 and 120')
        return v

app = FastAPI()

@app.post("/users/", response_model=User)
async def create_user(user: User):
    return user

# Test validation
valid_user = User(name="Alice", age=25, email="alice@example.com")
print(f"✅ Valid user: {valid_user.model_dump()}")
assert valid_user.model_dump() == {"name": "Alice", "age": 25, "email": "alice@example.com"}, valid_user

try:
    User(name="Bob", age=150)
except ValidationError as e:
    print(f"Rejected as expected: {e.errors()[0]['msg']}")
else:
    raise AssertionError("age > 120 accepted")

print("Example 2 completed")

## Example 3: Path Parameters and Query Parameters

In [ ]:
# Example 3: Path Parameters and Query Parameters
print("FastAPI - Path and Query Parameters")

from fastapi import FastAPI, Query, Path
from typing import Optional

app = FastAPI()

# Path parameters with validation
@app.get("/users/{user_id}")
async def get_user(user_id: int = Path(..., gt=0, description="User ID")):
    return {"user_id": user_id, "name": f"User {user_id}"}

# Query parameters with defaults and validation
@app.get("/items/")
async def list_items(
    skip: int = Query(0, ge=0, description="Items to skip"),
    limit: int = Query(10, ge=1, le=100, description="Items to return"),
    category: Optional[str] = Query(None, description="Filter by category")
):
    result = {"skip": skip, "limit": limit, "items": [f"item_{i}" for i in range(skip, skip + limit)]}
    if category:
        result["category"] = category
    return result

# Test parameter parsing
print(f"✅ User endpoint: /users/123 returns user info")
print(f"✅ Items endpoint: /items/?skip=0&limit=5&category=books returns filtered items")
print("Example 3 completed")

## Example 4: HTTP Methods and Status Codes

In [ ]:
# Example 4: HTTP Methods and Status Codes
print("FastAPI - HTTP Methods and Status Codes")

from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float

app = FastAPI()
items_db = {1: {"name": "Laptop", "price": 999.99}}

@app.post("/items/", status_code=status.HTTP_201_CREATED)
async def create_item(item: Item):
    item_id = max(items_db.keys()) + 1 if items_db else 1
    items_db[item_id] = item.model_dump()
    return {"id": item_id, **item.model_dump()}

@app.get("/items/{item_id}")
async def get_item(item_id: int):
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"id": item_id, **items_db[item_id]}

@app.delete("/items/{item_id}", status_code=status.HTTP_204_NO_CONTENT)
async def delete_item(item_id: int):
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail="Item not found")
    del items_db[item_id]

print(f"✅ CRUD endpoints created with proper status codes")
print(f"Database contains {len(items_db)} items")
print("Example 4 completed")

## Example 5: Dependency Injection and Background Tasks

In [ ]:
# Example 5: Dependency Injection and Background Tasks
print("FastAPI - Dependency Injection and Background Tasks")

from fastapi import FastAPI, Depends, BackgroundTasks
import time

app = FastAPI()

# Dependency function
def get_current_timestamp():
    return {"timestamp": time.time(), "formatted": time.strftime("%Y-%m-%d %H:%M:%S")}

# Background task function
def log_request(endpoint: str, timestamp: float):
    print(f"📝 Background task: Logged request to {endpoint} at {timestamp}")

# Route with dependency injection
@app.get("/time")
async def get_time(current_time: dict = Depends(get_current_timestamp)):
    return {"message": "Current time", **current_time}

# Route with background task
@app.post("/notify")
async def send_notification(
    message: str,
    background_tasks: BackgroundTasks,
    current_time: dict = Depends(get_current_timestamp)
):
    background_tasks.add_task(log_request, "/notify", current_time["timestamp"])
    return {"message": "Notification queued", "content": message}

print("✅ Dependency injection configured for timestamp")
print("✅ Background tasks enabled for logging")
print("Example 5 completed")

## Example 6: Requests through TestClient: status codes and JSON bodies

In [ ]:
# Example 6: Requests through TestClient
print("FastAPI - Requests through TestClient")

from fastapi import FastAPI, HTTPException, status
from fastapi.testclient import TestClient  # needs the httpx package
from pydantic import BaseModel, Field

class Item(BaseModel):
    name: str
    price: float = Field(gt=0)

app = FastAPI()
items_db = {1: {"name": "Laptop", "price": 999.99}}

@app.get("/items/{item_id}")
async def get_item(item_id: int):
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"id": item_id, **items_db[item_id]}

@app.post("/items/", status_code=status.HTTP_201_CREATED)
async def create_item(item: Item):
    item_id = max(items_db) + 1
    items_db[item_id] = item.model_dump()
    return {"id": item_id, **items_db[item_id]}

client = TestClient(app)
found = client.get("/items/1")
assert found.status_code == 200 and found.json() == {"id": 1, "name": "Laptop", "price": 999.99}, found.text
missing = client.get("/items/99")
assert missing.status_code == 404 and missing.json() == {"detail": "Item not found"}, missing.text
assert client.get("/items/abc").status_code == 422  # the path parameter must be an int

created = client.post("/items/", json={"name": "Mouse", "price": 25.5})
assert created.status_code == 201 and created.json() == {"id": 2, "name": "Mouse", "price": 25.5}, created.text
assert client.get("/items/2").json()["name"] == "Mouse"

invalid = client.post("/items/", json={"name": "Desk", "price": "not-a-number"})
assert invalid.status_code == 422 and invalid.json()["detail"][0]["loc"] == ["body", "price"], invalid.text
assert client.post("/items/", json={"name": "Desk", "price": -1}).status_code == 422  # price must be > 0
print("OK -- GET returns 200 with the item and 404 for a missing one; POST returns 201, and 422 for an invalid body")
print("Example 6 completed")